In [11]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine
from config.config import load_config
from sqlalchemy import text


# Определяем корень проекта: если мы в notebooks/, поднимаемся на уровень выше
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

# Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: C:\Users\utrop\OneDrive\Desktop\Python\Project_1\Cohort_RFM


In [16]:

config = load_config()

# Достаём настройки БД
db = config.database

# Создаём engine с явным указанием драйвера
engine = create_engine(f"postgresql+psycopg2://{db.user}:{db.password}@{db.host}:{db.port}/{db.dbname}")

In [19]:
sql_path = project_root / "SQL_Scripts" / "Cohort_ARPPU.sql"   # замените на точное имя файла
print(sql_path.exists())
print(sql_path.read_text(encoding="utf-8")[:500])   # первые 500 символов

True
with cohort_cte as (
-- разделяем на когорты по первому числу месяца даты первой покупки для каждого пользователя
	select
		card,
		datetime::date,
		datetime::date - first_value(datetime::date) over(partition by card order by datetime) as date_diff,
		date_trunc('month', first_value(datetime::date) over(partition by card order by datetime))::date as cohort,
		summ_with_disc
	from
		checks
	where
		card like '2000%'
		and datetime::date between '2021-01-01' and now()
	order by
		card
)
select
	c


In [23]:

with open(sql_path, "r", encoding="utf-8") as f:
    sql = f.read()

df = pd.read_sql_query(text(sql), con=engine)

In [24]:
df.head()

,cohort,users_count,1-30_users,31-60_users,61-90_users,91-120_users,121-150_users,151-180_users,first_purchase,1-30_day,...,31-60_day,31-60_ARPPU,61-90_day,61-90_60_ARPPU,91-120_day,91-120_ARPPU,121-150_day,121-150_ARPPU,151-180_day,151-180_ARPPU
0,2021-08-01,686,212,195,195,187,148,169,928.02,359.09,...,360.27,1267.41,360.02,1266.53,369.06,1353.87,280.90,1302.01,354.51,1439.04
1,2021-09-01,627,202,177,140,144,140,115,1020.30,520.57,...,443.53,1571.14,330.36,1479.53,353.06,1537.28,343.91,1540.24,297.05,1619.57
2,2021-10-01,563,172,140,137,121,107,98,1071.85,484.40,...,398.16,1601.19,370.56,1522.82,394.98,1837.82,261.28,1374.79,192.89,1108.13
3,2021-11-01,498,160,102,106,107,92,90,1114.75,516.59,...,279.97,1366.90,318.45,1496.10,288.42,1342.37,249.39,1349.97,180.86,1000.73
4,2021-12-01,509,117,100,94,81,76,59,1070.18,317.48,...,250.12,1273.13,237.81,1287.72,175.19,1100.88,163.86,1097.45,120.87,1042.73


In [27]:
from pathlib import Path
import pandas as pd
from sqlalchemy import text

# Папки
sql_dir = project_root / "SQL_Scripts" / "cohort"
output_dir = project_root / "data"
output_dir.mkdir(exist_ok=True)

# Файлы, которые нужно выгрузить
files_to_export = [
    "Cohort_ARPPU.sql",
    "Cohort_LTV_analysis.sql",
]

for file_name in files_to_export:
    sql_file = sql_dir / file_name
    if not sql_file.exists():
        print(f"❌ {file_name}: файл не найден")
        continue

    print(f"📄 {file_name}")

    # Читаем SQL
    raw = sql_file.read_text(encoding="utf-8-sig").strip()
    if not raw:
        print("   ⚠ пустой файл\n")
        continue

    # Выполняем
    try:
        df = pd.read_sql_query(text(raw), con=engine)
    except Exception as e:
        print(f"   ❌ ошибка выполнения: {type(e).__name__}: {e}\n")
        continue

    # Сохраняем в Excel
    output_path = output_dir / f"{sql_file.stem}.xlsx"
    df.to_excel(output_path, index=False, engine="openpyxl")

    print(f"   ✅ {len(df)} строк × {len(df.columns)} колонок")
    print(f"   💾 {output_path.name}\n")

print("Готово.")

📄 Cohort_ARPPU.sql
   ✅ 10 строк × 21 колонок
   💾 Cohort_ARPPU.xlsx

📄 Cohort_LTV_analysis.sql
   ✅ 10 строк × 8 колонок
   💾 Cohort_LTV_analysis.xlsx

Готово.


In [28]:
from pathlib import Path
from sqlalchemy import text

sql_dir = project_root / "SQL_Scripts"
rfm_view_dir = sql_dir / "RFM_view"

# Порядок важен: каждый следующий view зависит от предыдущего
rfm_files = [
    "RFM_metrics.sql",
    "RFM_segmentation_thresholds.sql",
    "RFM_segments.sql",
    "RFM_segments_share.sql",
]

for name in rfm_files:
    path = rfm_view_dir / name
    if not path.exists():
        print(f"❌ {name}: не найден")
        continue

    raw = path.read_text(encoding="utf-8-sig")
    print(f"⚙ {name}")

    try:
        with engine.begin() as conn:
            conn.execute(text(raw))
        print(f"   ✅ view создан/обновлён")
    except Exception as e:
        print(f"   ❌ {type(e).__name__}: {e}")

⚙ RFM_metrics.sql
   ❌ ProgrammingError: (psycopg2.errors.SyntaxError) ОШИБКА:  ошибка синтаксиса (примерное положение: "or")
LINE 1: create view or replace view rfm_metrics as (
                    ^

[SQL: create view or replace view rfm_metrics as (
	with dates as (
		select
			datetime::date,
			summ_with_disc,
			count(*) over(partition by card) as frequency,
			max(datetime::date) over(partition by card) as last_purchase,
			MAX(datetime::date) over() as last_date,
			MAX(datetime::date) over() - max(datetime::date) over(partition by card) as recency,
			card
		from
			checks
		where
			card like '2000%%'
			and summ_with_disc > 0
			and datetime::date between '2000-01-01' and now()
	)
	select
		card,
		min(recency) as recency,
		max(frequency) as frequency,
		sum(summ_with_disc) as monetary
	from
		dates
	group by
		card
)
]
(Background on this error at: https://sqlalche.me/e/20/f405)
⚙ RFM_segmentation_thresholds.sql
   ✅ view создан/обновлён
⚙ RFM_segments.sql
   ✅ view создан